# 13 - Computer Networks deployment benchmark

Measures actual serialized state-dict bytes, nonzero counts, dense CPU batch latency, p95 latency, and throughput for the compression matrix without assuming that unstructured zeros create speedups.

**Safety:** this notebook writes only new files under `results/tables/comnet/` and does not overwrite archived manuscript result tables.

In [ ]:
# Colab/bootstrap cell: no tokens or credentials are required.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time
from pathlib import Path

REPO = Path('/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression')
if not REPO.exists():
    # Local/Jupyter fallback: run the notebook from the repository root.
    REPO = Path.cwd()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.config import CFG, PATHS, set_all_seeds
set_all_seeds(CFG['anchor_seed'])

OUT_TABLE = PATHS.tables('comnet')
OUT_TABLE.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs:', OUT_TABLE)


## Configuration
Run this notebook on the hardware you intend to report. A Colab CPU is an edge proxy, not a physical gateway.

In [ ]:
DATASET = 'ciciot2023'
ARCH = 'cnn1d'
ARCH_KW = {'channels': (64, 128)}
SEED = int(CFG['anchor_seed'])
REBUILD_MISSING = False
BATCH_SIZES = (1, 32, 256, 1024)
WARMUP = 25
REPEATS = 100

TORCH_CPU_THREADS = 1


In [ ]:
import numpy as np
import pandas as pd
import torch
torch.set_num_threads(TORCH_CPU_THREADS)

from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor
from src import compression
from src.comnet_audit import (benchmark_cpu_model, serialized_state_dict_bytes,
                              index_value_sparse_payload_bytes, environment_record,
                              write_json)

df = clean(load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = temporal_within_capture_split(df, SEED)
m0, le, scaler, feat_cols = load_anchor(DATASET, ARCH, 'M0', SEED, arch_kwargs=ARCH_KW)


## Load models without silently rebuilding experiments

In [ ]:
def get_cell(cell):
    if cell == 'M0': return m0, 'float32'
    if cell == 'int8': return compression.to_int8(m0, ARCH)[0], 'float32'
    if cell == 'float16': return compression.to_float16(m0), 'float16'
    kwargs = {'channels':(24,48)} if cell=='distillation' else ARCH_KW
    try:
        return load_anchor(DATASET,ARCH,cell,SEED,arch_kwargs=kwargs)[0], 'float32'
    except Exception as exc:
        if not REBUILD_MISSING:
            print('SKIP',cell,exc); return None,None
        if cell.startswith('prune'):
            amount=int(cell.replace('prune',''))/100
            return compression.prune_and_finetune(m0,df,DATASET,splits,SEED,amount,arch=ARCH)[0], 'float32'
        if cell=='distillation':
            return compression.distill(df,DATASET,splits,SEED,m0,
                                       student_kwargs={'channels':(24,48)},arch=ARCH)[0], 'float32'
        raise

models={}
for cell in ['M0','prune50','prune80','distillation','int8','float16']:
    model,dtype=get_cell(cell)
    if model is not None: models[cell]=(model,dtype)
print(models.keys())


## Actual serialization, nonzero counts, and CPU latency/throughput

In [ ]:
size_rows=[]; bench_rows=[]
for cell,(model,dtype) in models.items():
    visible_params=sum(p.numel() for p in model.parameters())
    visible_nonzero=sum(int((p.detach().cpu()!=0).sum()) for p in model.parameters())
    quantized_packed = (cell == 'int8')
    size_rows.append({
        'cell':cell,
        'visible_parameters':visible_params,
        'visible_nonzero_parameters':visible_nonzero,
        'visible_parameter_sparsity':(1-visible_nonzero/max(visible_params,1)) if not quantized_packed else np.nan,
        'torch_state_dict_bytes':serialized_state_dict_bytes(model),
        'index_value_sparse_payload_estimate_bytes':(
            index_value_sparse_payload_bytes(model) if not quantized_packed else np.nan
        ),
        'numeric_input_dtype':dtype,
        'parameter_count_note':(
            'quantized Linear weights are packed and not fully exposed by model.parameters()'
            if quantized_packed else 'ordinary parameter tensors'
        ),
    })
    tab=benchmark_cpu_model(model,len(feat_cols),batch_sizes=BATCH_SIZES,
                            warmup=WARMUP,repeats=REPEATS,dtype=dtype)
    tab.insert(0,'cell',cell); bench_rows.append(tab)

sizes=pd.DataFrame(size_rows); bench=pd.concat(bench_rows,ignore_index=True)
sizes.to_csv(OUT_TABLE/'deployment_size_and_sparsity.csv',index=False)
bench.to_csv(OUT_TABLE/'deployment_cpu_latency_throughput.csv',index=False)
env=environment_record(); env['configured_torch_cpu_threads']=TORCH_CPU_THREADS
write_json(OUT_TABLE/'deployment_environment.json',env)
display(sizes)
display(bench.round(4))


## Reporting guard

In [ ]:
print('Report dense PyTorch timing exactly as measured. Do not claim that unstructured zeros accelerate inference unless a sparse execution backend is separately benchmarked.')
print('RSS snapshots are process-level observations, not a profiler-grade peak-memory measurement.')
print('The index-value payload is an estimated storage representation, not an executable sparse artifact.')
